# Chapter 9, Exercise 2: Two diacritizers, one G2P, and where they disagree

> **How to use this notebook.** Open it in Google Colab (File > Upload notebook, or the Colab badge on the companion website), then run the cells from top to bottom. Runtime > Change runtime type lets you pick a GPU when one is recommended below. Everything else runs on the free CPU tier.

## The exercise

**Chapter 9, Exercise 2.** Using Python, process a short passage of undiacritized Arabic news text through two different open-source diacritizers, convert each diacritized output to a phoneme sequence (as in the companion notebook), and write a short comparative analysis of where the two diacritizers disagree and how those differences would change the synthesized speech.

**Note.** The book's companion G2P notebook was not included in the material supplied for these solutions, so this notebook contains its own small rule-based MSA grapheme-to-phoneme converter (documented below, with its simplifications). The two diacritizers are **CAMeL Tools** (MLE disambiguator over the CALIMA-MSA database, which selects a fully vocalized analysis per word) and **Mishkal** (a rule-and-lexicon based open-source diacritizer). The passage is a short news-style paragraph composed for this exercise; you can paste any undiacritized MSA text in its place.

## Requirements

No GPU is required; the notebook runs on Colab's free CPU runtime.

## Testing status

Executed end to end with CAMeL Tools 1.6.0 and Mishkal. The G2P is a simplified rule set written for this exercise; its output should be checked by a native reader before use in a synthesizer.


<a href="https://colab.research.google.com/github/arabic-speech-book/arabic-speech-book.github.io/blob/main/docs/solutions/Chapter_09_Exercise_02.ipynb" target="_blank" rel="noopener"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

In [1]:
!pip install -q camel-tools mishkal pandas
!camel_data -i disambig-mle-calima-msa-r13

No new packages will be installed.


## 1. The passage

In [2]:
TEXT = ("أعلن وزير التعليم عن خطة جديدة لتطوير المدارس في المملكة. "
        "وقال الوزير إن الخطة تشمل تدريب المعلمين وبناء عشر مدارس في المناطق الشمالية. "
        "ومن المتوقع أن يبدأ التنفيذ في العام المقبل.")
SENTENCES = [t.strip() for t in TEXT.split(".") if t.strip()]   # both tools are run sentence by sentence
for t in SENTENCES: print("-", t)

- أعلن وزير التعليم عن خطة جديدة لتطوير المدارس في المملكة
- وقال الوزير إن الخطة تشمل تدريب المعلمين وبناء عشر مدارس في المناطق الشمالية
- ومن المتوقع أن يبدأ التنفيذ في العام المقبل


## 2. Diacritizer A: CAMeL Tools (MLE disambiguator)

The disambiguator picks, for each word, the most likely full analysis; its `diac` field is the vocalized form including case endings.

In [3]:
import re, unicodedata, pandas as pd
from camel_tools.disambig.mle import MLEDisambiguator
from camel_tools.tokenizers.word import simple_word_tokenize
from camel_tools.utils.dediac import dediac_ar

mle = MLEDisambiguator.pretrained("calima-msa-r13")
sent_words = [simple_word_tokenize(t) for t in SENTENCES]
camel_sents = []
for ws in sent_words:
    disamb = mle.disambiguate(ws)
    camel_sents.append([d.analyses[0].analysis["diac"] if d.analyses else d.word for d in disamb])
for cs in camel_sents: print(" ".join(cs))

أَعْلَنَ وَزِيرُ التَعْلِيمِ عَن خُطَّةٍ جَدِيدَةٍ لِتَطْوِيرِ المَدارِسِ فِي المَمْلَكَةِ
وَقالَ الوَزِيرُ إِن الخُطَّةِ تَشْمَل تَدْرِيبِ المُعَلِّمِينَ وَبِناءَ عَشَرَ مَدارِسِ فِي المَناطِقِ الشَمالِيَّةِ
وَمِن المُتَوَقَّعِ أَنَّ يَبْدَأ التَنْفِيذِ فِي العامِّ المُقْبِلِ


## 3. Diacritizer B: Mishkal

In [4]:
import mishkal.tashkeel
tashkeel = mishkal.tashkeel.TashkeelClass()
mishkal_sents = []
for t in SENTENCES:
    out = tashkeel.tashkeel(t).replace("", " ").strip()
    mishkal_sents.append(simple_word_tokenize(out))
    print(out)
# align on the undiacritized words (both tools keep the word sequence)
for ws, cs, ms in zip(sent_words, camel_sents, mishkal_sents):
    assert [dediac_ar(w) for w in cs] == [dediac_ar(w) for w in ms] == ws, "word sequences differ; check tokenization"
words = [w for ws in sent_words for w in ws]
camel_words = [w for ws in camel_sents for w in ws]; mishkal_words = [w for ws in mishkal_sents for w in ws]
print(len(words), "words in", len(SENTENCES), "sentences")

أَعْلَنَ وَزِيرُ التَّعْلِيمِ عَن خُطَّةِ جَديدَةٍ لِتَطْوِيرَ الْمُدَارِسِ فِي الْمَمْلَكَةِ
وَقَالَ الْوَزِيرُ إِنّ الْخُطَّةِ تَشْمَلُ تَدْرِيبُ الْمُعَلِّمِينَ وَبِنَاءِ عُشُرِ مَدَارِسِ فِي الْمَنَاطِقِ الشَّمَالِيَّةِ
ومِن الْمُتَوَقَّعِ أَن يَبْدَأُ التَّنْفِيذُ فِي الْعَامِ الْمُقْبِلِ
31 words in 3 sentences


## 4. A small rule-based MSA grapheme-to-phoneme converter

Input: one fully diacritized word. Output: a phoneme string in a broad IPA-style notation. Rules implemented: consonant map; short vowels from fatḥa, kasra, ḍamma; long vowels (fatḥa + alif or alif maqṣūra = /aː/, kasra + yāʾ = /iː/, ḍamma + wāw = /uː/); diphthongs (fatḥa + yāʾ/wāw with sukūn = /aj/, /aw/); shadda = geminate (doubled symbol); tanwīn = /an/, /in/, /un/; hamza forms = /ʔ/; madda = /ʔaː/; **sun-letter assimilation** of the article (ال + sun letter with shadda gives a geminate and no /l/); **hamzat al-waṣl** (a bare word-initial alif is /ʔ/ plus its vowel at the start of a phrase and is dropped inside a phrase); tāʾ marbūṭa = /t/ when followed by a vowel, /a/ in pausal position. With `pausal=True` the final short vowel or tanwīn of the last word of each sentence is dropped (waqf). Simplifications: no cross-word vowel elision beyond waṣl, no dagger alif, no special reading of الله.

In [5]:
CONS = {"ب":"b","ت":"t","ث":"θ","ج":"dʒ","ح":"ħ","خ":"x","د":"d","ذ":"ð","ر":"r","ز":"z","س":"s","ش":"ʃ",
        "ص":"sˤ","ض":"dˤ","ط":"tˤ","ظ":"ðˤ","ع":"ʕ","غ":"ɣ","ف":"f","ق":"q","ك":"k","ل":"l","م":"m","ن":"n",
        "ه":"h","ء":"ʔ","أ":"ʔ","إ":"ʔ","ؤ":"ʔ","ئ":"ʔ","ٱ":"ʔ"}
SUN = set("تثدذرزسشصضطظلن")
FATHA, DAMMA, KASRA, SUKUN, SHADDA = "\u064E", "\u064F", "\u0650", "\u0652", "\u0651"
TANWIN = {"\u064B": "an", "\u064C": "un", "\u064D": "in"}
SHORT = {FATHA: "a", DAMMA: "u", KASRA: "i"}
DIAC_SET = set(SHORT) | set(TANWIN) | {SUKUN, SHADDA}

VOWELS = {"a", "i", "u", "aː", "iː", "uː", "an", "in", "un", "aj", "aw"}

def g2p_word(w, phrase_initial=True, pausal=False):
    w = unicodedata.normalize("NFC", w)
    toks = []                                  # [letter, diacritics that follow it]
    for ch in w:
        if ch in DIAC_SET and toks: toks[-1][1] += ch
        elif ch not in DIAC_SET: toks.append([ch, ""])
    ph, i, force_gem = [], 0, False
    while i < len(toks):
        L, d = toks[i]
        nxt = toks[i+1][0] if i+1 < len(toks) else None
        last_is_vowel = bool(ph) and ph[-1] in VOWELS
        # --- definite article: bare alif + lam at the start of the word (hamzat al-wasl) ---
        if i == 0 and L == "ا" and nxt == "ل" and i + 2 < len(toks):
            if phrase_initial: ph.append("ʔ"); ph.append("a")
            if toks[i+2][0] in SUN:
                force_gem = True                 # sun letter: /l/ assimilates, the sun letter is geminated
            else:
                ph.append("l")
            i += 2; continue
        if i == 0 and L == "ا":                  # other hamzat al-wasl (e.g. اسم, verb forms)
            if phrase_initial: ph.append("ʔ"); ph.append(SHORT.get(d[:1], "i") if d else "i")
            i += 1; continue
        if L == "آ": ph.append("ʔ"); ph.append("aː"); i += 1; continue
        if L == "ة":                             # ta marbuta
            if d and not pausal:
                ph.append("t"); ph.append(SHORT.get(d[0], "") or TANWIN.get(d[0], ""))
            else:
                ph.append("a")
            i += 1; continue
        if L == "ا":                             # alif: lengthens a preceding /a/, or is /aː/ after a bare consonant
            if ph and ph[-1] == "a": ph[-1] = "aː"
            elif ph and not last_is_vowel: ph.append("aː")
            i += 1; continue
        if L == "ى":
            if ph and ph[-1] == "a": ph[-1] = "aː"
            else: ph.append("aː")
            i += 1; continue
        if L in "يو" and not any(x in d for x in SHORT) and SHADDA not in d and not any(x in d for x in TANWIN):
            # long vowel or diphthong (a bare ya/waw after a vowel or a bare consonant)
            if ph and ph[-1] == "i" and L == "ي": ph[-1] = "iː"; i += 1; continue
            if ph and ph[-1] == "u" and L == "و": ph[-1] = "uː"; i += 1; continue
            if ph and ph[-1] == "a" and (SUKUN in d or not d): ph[-1] = "a" + ("j" if L == "ي" else "w"); i += 1; continue
            if ph and not last_is_vowel and not d: ph.append("iː" if L == "ي" else "uː"); i += 1; continue
        base = CONS.get(L, "j" if L == "ي" else "w" if L == "و" else "")
        if SHADDA in d or force_gem: base = base + "ː"; force_gem = False
        ph.append(base)
        for c in d:
            if c in SHORT: ph.append(SHORT[c])
            elif c in TANWIN: ph.append(TANWIN[c])
        i += 1
    if pausal and ph and ph[-1] in {"a", "i", "u", "an", "in", "un"}:
        ph.pop()                                 # waqf: drop the final short vowel or tanwin
        if ph and ph[-1] == "t" and w.rstrip("".join(DIAC_SET)).endswith("ة"): ph[-1] = "a"
    return "".join(ph)

def g2p_text(sents, pausal_final=True):
    # sents: list of sentences, each a list of diacritized words. Returns one phoneme string per word.
    out = []
    for ws in sents:
        for k, w in enumerate(ws):
            out.append(g2p_word(w, phrase_initial=(k == 0), pausal=(pausal_final and k == len(ws)-1)))
    return out

for test in ["الشَّمْسُ", "الشمس", "الْقَمَرُ", "مَدْرَسَةٌ", "عَلَّمَ", "كِتَابٌ", "بَيْتٌ", "يَوْمٌ", "المَدارِسِ", "جَديدَةٍ", "اللُّغَةُ"]:
    print(test, "->", g2p_word(test, pausal=False), "| pausal:", g2p_word(test, pausal=True))

الشَّمْسُ -> ʔaʃːamsu | pausal: ʔaʃːams
الشمس -> ʔaʃːms | pausal: ʔaʃːms
الْقَمَرُ -> ʔalqamaru | pausal: ʔalqamar
مَدْرَسَةٌ -> madrasatun | pausal: madrasa
عَلَّمَ -> ʕalːama | pausal: ʕalːam
كِتَابٌ -> kitaːbun | pausal: kitaːb
بَيْتٌ -> bajtun | pausal: bajt
يَوْمٌ -> jawmun | pausal: jawm
المَدارِسِ -> ʔalmadaːrisi | pausal: ʔalmadaːris
جَديدَةٍ -> dʒadiːdatin | pausal: dʒadiːda
اللُّغَةُ -> ʔalːuɣatu | pausal: ʔalːuɣa


## 5. Phoneme sequences from both diacritizers, and where they disagree

In [6]:
ph_camel = g2p_text(camel_sents); ph_mishkal = g2p_text(mishkal_sents)
rows = []
for w, a, b, pa, pb in zip(words, camel_words, mishkal_words, ph_camel, ph_mishkal):
    rows.append({"word": w, "CAMeL": a, "Mishkal": b, "phon CAMeL": pa, "phon Mishkal": pb,
                 "diac same?": a == b, "phon same?": pa == pb})
df = pd.DataFrame(rows)
n = len(df)
print(f"words: {n} | diacritization disagreements: {(~df['diac same?']).sum()} "
      f"({100*(~df['diac same?']).mean():.0f} %) | phoneme-sequence disagreements: {(~df['phon same?']).sum()} "
      f"({100*(~df['phon same?']).mean():.0f} %)")
pd.set_option("display.max_colwidth", 40)
df[~df["diac same?"]]

words: 31 | diacritization disagreements: 25 (81 %) | phoneme-sequence disagreements: 13 (42 %)


,word,CAMeL,Mishkal,phon CAMeL,phon Mishkal,diac same?,phon same?
2,التعليم,التَعْلِيمِ,التَّعْلِيمِ,tːaʕliːmi,tːaʕliːmi,False,True
4,خطة,خُطَّةٍ,خُطَّةِ,xutˤːatin,xutˤːati,False,False
5,جديدة,جَدِيدَةٍ,جَديدَةٍ,dʒadiːdatin,dʒadiːdatin,False,True
6,لتطوير,لِتَطْوِيرِ,لِتَطْوِيرَ,litatˤwiːri,litatˤwiːra,False,False
7,المدارس,المَدارِسِ,الْمُدَارِسِ,lmadaːrisi,lmudaːrisi,False,False
9,المملكة,المَمْلَكَةِ,الْمَمْلَكَةِ,lmamlaka,lmamlaka,False,True
10,وقال,وَقالَ,وَقَالَ,waqaːla,waqaːla,False,True
11,الوزير,الوَزِيرُ,الْوَزِيرُ,lwaziːru,lwaziːru,False,True
12,إن,إِن,إِنّ,ʔin,ʔinː,False,False
13,الخطة,الخُطَّةِ,الْخُطَّةِ,lxutˤːati,lxutˤːati,False,True


In [7]:
# Where do they disagree? Compare the PHONEME strings, because the two tools write diacritics with different
# conventions (CAMeL omits the fatha before alif and the shadda on sun letters; Mishkal writes them), and those
# conventions do not change the pronunciation.
def strip_final_vowel(p):
    return re.sub(r"(an|in|un|[aiu])$", "", p)
diff = df[~df["phon same?"]].copy()
diff["only final vowel / case ending"] = [strip_final_vowel(a) == strip_final_vowel(b) for a, b in zip(diff["phon CAMeL"], diff["phon Mishkal"])]
print("phoneme-level disagreements:", len(diff))
print("  of which only in the final short vowel (case or mood ending):", int(diff["only final vowel / case ending"].sum()))
print("  of which inside the word (stem vowels, gemination, dropped vowel):", int((~diff["only final vowel / case ending"]).sum()))
print("\nFull phoneme string, CAMeL   :", " ".join(ph_camel))
print("Full phoneme string, Mishkal :", " ".join(ph_mishkal))
diff[["word", "CAMeL", "Mishkal", "phon CAMeL", "phon Mishkal", "only final vowel / case ending"]]

phoneme-level disagreements: 13
  of which only in the final short vowel (case or mood ending): 7
  of which inside the word (stem vowels, gemination, dropped vowel): 6

Full phoneme string, CAMeL   : ʔaʕlana waziːru tːaʕliːmi ʕan xutˤːatin dʒadiːdatin litatˤwiːri lmadaːrisi fiː lmamlaka waqaːla lwaziːru ʔin lxutˤːati taʃmal tadriːbi lmuʕalːimiːna wabinaːʔa ʕaʃara madaːrisi fiː lmanaːtˤiqi ʃːamaːlijːa wamin lmutawaqːaʕi ʔanːa jabdaʔ tːanfiːði fiː lʕaːmːi lmuqbil
Full phoneme string, Mishkal : ʔaʕlana waziːru tːaʕliːmi ʕan xutˤːati dʒadiːdatin litatˤwiːra lmudaːrisi fiː lmamlaka waqaːla lwaziːru ʔinː lxutˤːati taʃmalu tadriːbu lmuʕalːimiːna wabinaːʔi ʕuʃuri madaːrisi fiː lmanaːtˤiqi ʃːamaːlijːa wmin lmutawaqːaʕi ʔan jabdaʔu tːanfiːðu fiː lʕaːmi lmuqbil


,word,CAMeL,Mishkal,phon CAMeL,phon Mishkal,only final vowel / case ending
4,خطة,خُطَّةٍ,خُطَّةِ,xutˤːatin,xutˤːati,True
6,لتطوير,لِتَطْوِيرِ,لِتَطْوِيرَ,litatˤwiːri,litatˤwiːra,True
7,المدارس,المَدارِسِ,الْمُدَارِسِ,lmadaːrisi,lmudaːrisi,False
12,إن,إِن,إِنّ,ʔin,ʔinː,False
14,تشمل,تَشْمَل,تَشْمَلُ,taʃmal,taʃmalu,True
15,تدريب,تَدْرِيبِ,تَدْرِيبُ,tadriːbi,tadriːbu,True
17,وبناء,وَبِناءَ,وَبِنَاءِ,wabinaːʔa,wabinaːʔi,True
18,عشر,عَشَرَ,عُشُرِ,ʕaʃara,ʕuʃuri,False
23,ومن,وَمِن,ومِن,wamin,wmin,False
25,أن,أَنَّ,أَن,ʔanːa,ʔan,False


## 6. Comparative analysis (template; fill from the tables above)

Group the disagreements into three kinds and say what a listener would hear:

1. **Case and mood endings only** (for example خُطَّةٍ vs خُطَّةَ). In *connected* synthesis these change the final short vowel of the word (/-tin/ vs /-ta/): audible, grammatically meaningful, but the word is still recognized. In *pausal* synthesis at a phrase end they vanish entirely, so the two outputs sound identical there. Section 2.7 notes that these endings are the hard part for diacritizers, and this is where the two tools disagree most.
2. **Internal vowels or shadda** (for example المَدارِس vs المُدَارِس, or a verb read as active vs passive). These change the *stem* and can change the word: a wrong internal vowel is a mispronunciation, and sometimes a different lexeme (Section 1.5.2, عَلِمَ / عَلَّمَ / عِلْم). A synthesizer renders whichever it is given fluently, so the error is not signalled by the audio.
3. **Definite article and waṣl handling** (whether the alif of ال carries a hamza or a waṣl mark, sun-letter shadda present or absent). If the diacritizer omits the shadda on a sun letter, a naive G2P will pronounce the /l/ (al-shams instead of ash-shams), which is exactly the rule the book says a G2P must encode (Section 2.6); the G2P here checks for the shadda, so a missing shadda changes the phoneme output.

Report the counts from Section 5 (how many words differ, how many only in the case ending, how many internally), quote two or three concrete words for each kind with both phoneme strings, and state which tool you would trust for each kind and why. Keep in mind that neither output has been checked by a human: to score either diacritizer you would need a human-verified reference and a diacritic error rate (DiacER), Section 2.7.